<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network 徽标">
    </a>
</p>


# 实验：支持向量机 vs 普通线性分类器


<h2>目录</h2>
<p>我们将对 sklearn 库中流行的手写数据集进行分类，并比较逻辑回归和 SVM 的结果。在 Sklearn 库中，有多种方法可以将逻辑回归用于多分类任务；在本实验中，我们将使用 `multinomial` 选项，这类似于我们之前讨论的 Softmax 函数。</p>


- [可视化数据集中的一些手写图片](#Visualize-Some-Handwritten-Images-in-the-Dataset)
- [使用逻辑回归进行手写分类](#Hand-written-classification-with-Logistic-Regression)
- [使用 SVM 进行手写分类](#Hand-Written-Classification-with-SVM)
- [使用 K 折交叉验证比较 SVM 与逻辑回归](#Comparing-both-SVM-and-Logistic-Regression-with-K-Fold-Cross-Validation)

<p>预计所需时间：<strong>60 分钟</strong></p>

<hr>


## 加载重要库和数字数据集


**安装可能需要一些时间，请耐心等待……**


In [ ]:
!pip3 install torch torchvision torchaudio
!pip install matplotlib
!pip install scikit-learn
!pip install pandas
!pip install seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets, svm, metrics, model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [ ]:
digits = datasets.load_digits()

In [ ]:
target = digits.target
flatten_digits = digits.images.reshape((len(digits.images), -1))
print(f"flatten_digits shape: {flatten_digits.shape}")

## 可视化数据集中的一些手写图片


In [ ]:
_, axes = plt.subplots(nrows=1, ncols=5, figsize=(10, 4))
for ax, image, label in zip(axes, digits.images, target):
    ax.set_axis_off()
    ax.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
    ax.set_title('%i' % label)

## 将图片划分为训练集和测试集


我将测试集大小设置为总数据集的 20%


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(flatten_digits, target, test_size=0.2)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 使用逻辑回归进行手写分类


对数据集进行标准化，使所有变量特征处于相同尺度


In [ ]:
scaler = StandardScaler()
X_train_logistic = scaler.fit_transform(X_train)
X_test_logistic = scaler.transform(X_test)

创建逻辑回归并拟合，使用 <code>l1</code> 惩罚项。请注意，由于这是一个多分类问题，逻辑回归参数 `multi_class` 设置为 `multinomial`。


In [ ]:
logit = LogisticRegression(C=0.01, penalty='l1', solver='saga', tol=0.1)

In [ ]:
logit.fit(X_train_logistic, y_train)

In [ ]:
y_pred_logistic = logit.predict(X_test_logistic)

获取逻辑回归的准确率


In [ ]:
print("Accuracy: "+str(accuracy_score(y_pred_logistic, y_test)))

让我们绘制混淆矩阵，矩阵的每一行代表预测类别中的实例，每一列代表真实类别中的实例。


In [ ]:
label_names = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
cmx = confusion_matrix(y_test, y_pred_logistic, labels=label_names)

准确率不错，超过 80%，但我们可以看到一些严重误分类的值，分类器很难正确分类 <code>8</code>


In [ ]:
df_cm = pd.DataFrame(cmx)
# 设置图像大小为 (10,7)
sns.set(font_scale=1.4) # 标签大小
sns.heatmap(df_cm, annot=True, annot_kws={"size": 16}) # 字体大小
title = "Confusion Matrix for SVM results"
plt.title(title)
plt.show()

## 使用 SVM 进行手写分类


创建并拟合 SVM 模型


In [ ]:
svm_classifier = svm.SVC(gamma='scale')

In [ ]:
svm_classifier.fit(X_train, y_train)

对测试集进行预测


In [ ]:
y_pred_svm = svm_classifier.predict(X_test)

获取 SVM 模型的准确率，可以看到我们得到了一个几乎完美的模型


In [ ]:
print("Accuracy: "+str(accuracy_score(y_test, y_pred_svm)))

让我们查看 SVM 的混淆矩阵，可以看到 SVM 的模型几乎完美


In [ ]:
label_names = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
cmx = confusion_matrix(y_test, y_pred_svm, labels=label_names)

In [ ]:
df_cm = pd.DataFrame(cmx)
# 设置图像大小为 (10,7)
sns.set(font_scale=1.4) # 标签大小
sns.heatmap(df_cm, annot=True, annot_kws={"size": 16}) # 字体大小
title = "Confusion Matrix for SVM results"
plt.title(title)
plt.show()

## 使用 K 折交叉验证比较 SVM 与逻辑回归

K 折交叉验证用于样本有限的情况，手写数据集约有 1800 个样本，这将使所有数据在不同时间有机会分别进入训练集和测试集。我们将添加 <code>l2</code> 正则化，以可视化它们与 SVM 相比的表现。


In [ ]:
algorithm = []
algorithm.append(('SVM', svm_classifier))
algorithm.append(('Logistic_L1', logit))
algorithm.append(('Logistic_L2', LogisticRegression(C=0.01, penalty='l2', solver='saga', tol=0.1)))


results = []
names = []
y = digits.target
for name, algo in algorithm:
    k_fold = model_selection.KFold(n_splits=10,shuffle=True, random_state=10)
    if name == 'SVM':
        X = flatten_digits
        cv_results = model_selection.cross_val_score(algo, X, y, cv=k_fold, scoring='accuracy')
    else:
        scaler = StandardScaler()
        X = scaler.fit_transform(flatten_digits)
        cv_results = model_selection.cross_val_score(algo, X, y, cv=k_fold, scoring='accuracy')
        
    results.append(cv_results)
    names.append(name)

我们绘制结果，可以看到即使使用 K 折交叉验证，SVM 始终表现更好，平均而言也优于两种逻辑回归


In [ ]:
fig = plt.figure()
fig.suptitle('Compare Logistic and SVM results')
ax = fig.add_subplot()
plt.boxplot(results)
plt.ylabel('Accuracy')
ax.set_xticklabels(names)
plt.show()

## 参考资料


1.  [识别手写数字](https://scikit-learn.org/stable/auto_examples/classification/plot_digits_classification.html?utm_email=Email&utm_source=Nurture&utm_content=000026UJ&utm_term=10006555&utm_campaign=PLACEHOLDER&utm_id=SkillsNetwork-Courses-IBMDeveloperSkillsNetwork-CV0101EN-Coursera-25797139)
2.  [使用多项式逻辑回归 + L1 进行 MNIST 分类](https://scikit-learn.org/stable/auto_examples/linear_model/plot_sparse_logistic_regression_mnist.html?utm_email=Email&utm_source=Nurture&utm_content=000026UJ&utm_term=10006555&utm_campaign=PLACEHOLDER&utm_id=SkillsNetwork-Courses-IBMDeveloperSkillsNetwork-CV0101EN-Coursera-25797139)


<h2>作者</h2>


 [Aije Egwaikhide](https://www.linkedin.com/in/aije-egwaikhide/?utm_email=Email&utm_source=Nurture&utm_content=000026UJ&utm_term=10006555&utm_campaign=PLACEHOLDER&utm_id=SkillsNetwork-Courses-IBMDeveloperSkillsNetwork-CV0101EN-Coursera-25797139) 是 IBM 的数据科学家，拥有曼尼托巴大学经济学与统计学学位，以及金斯敦圣劳伦斯学院商业分析研究生文凭。她目前正在女王大学攻读管理分析硕士学位。她是 IBM Developer Skills Network 小组的成员，将自己现实世界的经验带到她创建的课程中。


# 参考资料


[1]  <a href='https://opencv.org/'>Open CV</a>


<!--<h2>Change Log</h2>-->


<!--<table>
    <tr>
        <th>日期（YYYY-MM-DD）</th>
        <th>版本</th>
        <th>修改者</th>
        <th>变更说明</th>
    </tr>
    <tr>
        <td>2025-07-10</td>
        <td>0.1</td>
        <td>Sathya</td>
        <td>将实验转换为 JupyterLab Current 版本</td>
    </tr>
    <tr>
        <td>2021-03-30</td>
        <td>0.1</td>
        <td>Aije</td>
        <td>创建实验的原始版本</td>
    </tr>
</table>-->


## <h3 align="center"> © IBM Corporation。保留所有权利。 <h3/>
